# Background Information #

information

## Imports ##

In [ ]:
# installing required libraries
!pip install biopython scikit-bio

In [ ]:
from Bio import SeqIO
from Bio.Align import PairwiseAligner

## Load Sequences

In [ ]:
#loading genomes - reading FASTA files into SeqRecord objects
records = list(SeqIO.parse('genomes.fasta', 'fasta')) # reads and stores DNA sequences from FASTA file
# note: genomes.fasta file hasn't been created yet
sequences = [rec.seq for rec in records] #extracts the DNA sequence

In [ ]:
#checking that everything loaded correctly
print(f"Number of sequences: {len(sequences)}")

for record in records:
    print(record.id, len(record.seq))

## Validate Sequences

In [16]:
# Checks whether an input contains only A, T, C, and G.
def is_valid_dna(sequence):
    valid_bases = set('ATCG')
    sequence = ''.join(sequence.split()).upper()

    if len(sequence) == 0:
        return False
        
    return all(base in valid_bases for base in sequence)

In [19]:
# testing function
print(is_valid_dna(" ATCG"))
print(is_valid_dna("ATXG"))
print(is_valid_dna(""))
print(is_valid_dna("atcgTga atta"))

True
False
False
True


## Align Sequences

In [ ]:
# perform sequence alignment: creates an aligner, compares first 2 sequences in the FASTA file, 
# returns possible alignments, displays the best-scoring alignment

aligner = PairwiseAligner()

alignment = aligner.align(sequences[0], sequences[1])

print(alignment[0])

## Identify Differences

In [ ]:
# detect mutations/variants. finds substitutions, insertions, or deletions
def find_mutations(seq1, seq2):
    mutations = []
    for i, (b1, b2) in enumerate(zip(seq1, seq2)):
        if b1 != b2:
            mutations.append({'pos': i, 'ref': b1, 'alt': b2})
    return mutations

mut_list = find_mutations(sequences[0], sequences[1])
for m in mut_list:
    print(f"Pos {m['pos']}:  {m['ref']} \rightarrow {m['alt']}")

## Load Risk Database

In [ ]:
# Loads a database containing DNA sequences, virus platforms, and associated risk factors.
def load_risk_database(filename):
    risk_database = []
    try:
        with open(filename, 'r') as file:
            for line in file:
                line = line.strip()
                if line and not line.startswith('#'):
                    parts = line.split(',', 2)
                    if len(parts) == 3:
                        sequence, platform, risk = parts
                        risk_database.append({
                            'sequence': sequence.strip().upper(),
                            'platform': platform.strip(),
                            'risk': risk.strip()
                        })
    except FileNotFoundError:
        print(f"Error: Could not find database file '{filename}'.")
    return risk_database

## Screen Sequences

## Generate Report

In [ ]:
# Checks whether an input contains only A, T, C, and G.
def is_valid_dna(sequence):
    valid_bases = set('ATCG')
    return all(base in valid_bases for base in sequence.upper())

In [ ]:
# Compares an input sequence against sequences in the risk database.
def screen_sequence(input_sequence, risk_database):
    input_sequence = input_sequence.strip().upper()
    matches = []
    for entry in risk_database:
        db_sequence = entry['sequence']
        if input_sequence == db_sequence:
            matches.append({**entry, 'match_type': 'exact'})
        elif db_sequence in input_sequence:
            matches.append({**entry, 'match_type': 'partial (input contains known risk sequence)'})
        elif input_sequence in db_sequence:
            matches.append({**entry, 'match_type': 'partial (input is part of known risk sequence)'})
    return matches

In [ ]:
# Prints a report identifying any matches and their associated risk factors.
def generate_report(matches, input_sequence):
    print("\n--- Oncolytic Virus Risk Screening Report ---")
    print(f"Input sequence: {input_sequence}")
    if not matches:
        print("Result: No known risk factors found for this sequence.")
    else:
        print(f"Result: {len(matches)} potential risk factor(s) identified:\n")
        for i, match in enumerate(matches, start=1):
            print(f"  {i}. Platform: {match['platform']}")
            print(f"     Match type: {match['match_type']}")
            print(f"     Risk factor: {match['risk']}\n")
    print("--- End of Report ---\n")

In [ ]:
# risk_database = load_risk_database( don't have a risk database yet )
print(f"Loaded {len(risk_database)} risk entries.")

In [ ]:
test_sequence = # insert chosen sequence. can make a loop to test multiple sequences if desired

In [ ]:
if is_valid_dna(test_sequence):
    matches = screen_sequence(test_sequence, risk_database)
    generate_report(matches, test_sequence)
else:
    print("Invalid input: sequence must only contain A, T, C, and G.")